# Assignment 5: Employee Attrition Prediction — Decision Tree vs Random Forest

**Dataset:** IBM HR Analytics Employee Attrition & Performance
**Source:** https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset

**Objective:** Predict employee attrition using Decision Tree and Random Forest classifiers, and compare their performance.

> **Before running:** Download `WA_Fn-UseC_-HR-Employee-Attrition.csv` from the Kaggle link above and place it in the `data/` folder of this repository (it is not included here — see README for details on why).


In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42


## Task 1: Data Understanding

1. Load the dataset using Pandas
2. Display the first five records
3. Identify numerical features, categorical features, and the target variable
4. Display dataset info and summary statistics


In [ ]:
# 1. Load the dataset
DATA_PATH = "data/WA_Fn-UseC_-HR-Employee-Attrition.csv"
df = pd.read_csv(DATA_PATH)
print("Shape of dataset:", df.shape)


In [ ]:
# 2. Display the first five records
df.head()


In [ ]:
# 3. Identify numerical and categorical features, and the target variable
target_variable = "Attrition"

numerical_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = df.select_dtypes(include=["object"]).columns.tolist()

# Target is categorical but we'll treat it separately
if target_variable in categorical_features:
    categorical_features.remove(target_variable)

print(f"Target variable: {target_variable}\n")
print(f"Numerical features ({len(numerical_features)}):\n{numerical_features}\n")
print(f"Categorical features ({len(categorical_features)}):\n{categorical_features}")


In [ ]:
# 4. Dataset info and summary statistics
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
# Target distribution
df[target_variable].value_counts()


## Task 2: Data Preprocessing

- Check for missing values
- Remove unnecessary columns (constant / non-informative columns)
- Encode categorical variables
- Split the dataset into 80% training and 20% testing


In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing if not missing.empty else "None — no missing values found.")


In [ ]:
# Remove unnecessary / constant columns
# EmployeeCount, StandardHours, and Over18 are constant across all rows in this dataset
# EmployeeNumber is a unique identifier with no predictive value
cols_to_drop = [c for c in ["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"] if c in df.columns]
df_clean = df.drop(columns=cols_to_drop)
print(f"Dropped columns: {cols_to_drop}")
print("New shape:", df_clean.shape)


In [ ]:
# Encode categorical variables using Label Encoding
df_encoded = df_clean.copy()
label_encoders = {}

categorical_cols = df_encoded.select_dtypes(include=["object"]).columns.tolist()
print("Categorical columns to encode:", categorical_cols)

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

df_encoded.head()


In [ ]:
# Separate features and target
X = df_encoded.drop(columns=[target_variable])
y = df_encoded[target_variable]  # 1 = Yes (attrition), 0 = No

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


## Task 3: Model Development

- Model 1: Decision Tree Classifier
- Model 2: Random Forest Classifier (100 estimators)

Both models are trained on the same training set and used to predict on the same test set.


In [ ]:
# Model 1: Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)


In [ ]:
# Model 2: Random Forest Classifier (100 estimators)
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)


## Task 4: Model Evaluation and Comparison

Evaluate both models using Accuracy, Precision, Recall, and F1-Score.
Also generate confusion matrices for both models and a feature importance plot for Random Forest.


In [ ]:
def evaluate_model(name, y_true, y_pred):
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred),
    }
    return metrics

dt_metrics = evaluate_model("Decision Tree", y_test, dt_preds)
rf_metrics = evaluate_model("Random Forest", y_test, rf_preds)

results_df = pd.DataFrame([dt_metrics, rf_metrics]).set_index("Model")
results_df.round(4)


In [ ]:
print("Decision Tree — Classification Report")
print(classification_report(y_test, dt_preds, target_names=["No Attrition", "Attrition"]))

print("\nRandom Forest — Classification Report")
print(classification_report(y_test, rf_preds, target_names=["No Attrition", "Attrition"]))


In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm_dt = confusion_matrix(y_test, dt_preds)
disp_dt = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=["No", "Yes"])
disp_dt.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Decision Tree — Confusion Matrix")

cm_rf = confusion_matrix(y_test, rf_preds)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=["No", "Yes"])
disp_rf.plot(ax=axes[1], cmap="Greens", colorbar=False)
axes[1].set_title("Random Forest — Confusion Matrix")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()


In [ ]:
# Feature Importance plot for Random Forest
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 7))
sns.barplot(x=top_features.values, y=top_features.index, palette="viridis")
plt.title("Top 15 Feature Importances — Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()


### Observations

*(Fill these in after running the notebook on the real dataset — sample talking points below; replace with your actual numbers.)*

1. **Accuracy:** Random Forest typically achieves higher accuracy than a single Decision Tree because it aggregates predictions across many trees, reducing variance.
2. **Precision vs Recall on the minority class:** Since attrition (`Yes`) is the minority class in this dataset, both models often show lower recall than accuracy — check which model catches more actual leavers (higher recall) versus which raises fewer false alarms (higher precision).
3. **Overfitting:** The Decision Tree usually shows a bigger gap between training and test performance (more prone to overfitting on the training data) compared to the Random Forest, which generalizes better due to bagging and feature randomness.
4. **Feature importance:** Features like `OverTime`, `MonthlyIncome`, `Age`, `TotalWorkingYears`, and `JobSatisfaction` tend to be among the top predictors of attrition — confirm this against your actual output above.


## Task 5: Conclusion

*(150–200 words — edit this after reviewing your actual results above.)*

Based on the evaluation metrics, the **Random Forest Classifier** generally outperformed the single **Decision Tree Classifier** across accuracy, precision, recall, and F1-score, and its confusion matrix showed fewer misclassifications overall. This is expected: Random Forest is an ensemble method that builds many decision trees on bootstrapped samples of the data with random subsets of features at each split, then averages their predictions. This process, known as bagging combined with feature randomness, reduces the variance and overfitting that a single Decision Tree is prone to, while still capturing complex, non-linear relationships in the data.

A key limitation of the Decision Tree is that it tends to overfit the training data, especially when grown without depth restrictions, which hurts its ability to generalize to unseen employees. A key limitation of Random Forest is that it is far less interpretable than a single tree — while feature importance scores give some insight, the actual decision logic is distributed across 100 trees and cannot be visualized or explained as a simple rule set, and it is also more computationally expensive to train and predict with.

Overall, for this attrition prediction task, Random Forest is the more reliable choice when accuracy and generalization matter more than interpretability.
